# 04 · Results: what each sensor and learning add, with intervals

The headline comparisons of `results/report.md`: every forecast against the radar's pysteps
extrapolation, with 95% intervals from resampling storm days (paired bootstrap).

In [1]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")       # torch + pysteps in one process (macOS)
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
ROOT = Path.cwd().resolve()
while not (ROOT / "core").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "projects" / "nowcasting" / "multisensor" / "src"))
RES = ROOT / "projects" / "nowcasting" / "multisensor" / "results"
from multisensor_nowcasting.settings import NETWORKS
from multisensor_nowcasting.cube import Cube
from multisensor_nowcasting.report import label
NETS = [n for n in NETWORKS if Cube.exists(n)]
plt.rcParams.update({"figure.dpi": 72, "axes.titlesize": 10})
pd.set_option("display.precision", 3)
pd.set_option("display.width", 160)
NETS = [n for n in NETS if (RES / f"scores_{n}.csv").exists()]
print("networks with results:", NETS)

networks with results: ['openmrg', 'openmesh']


In [2]:
from multisensor_nowcasting.report import KEY
for net in NETS:
    hl = pd.read_csv(RES / f"headline_{net}.csv")
    for metric, lm, region in (("CSI_1", 60, "domain"), ("gauge next-hour RMSE", 60, "gauges")):
        q = hl[(hl.metric == metric) & (hl.lead_min == lm) & (hl.region == region) & hl.forecast.isin(KEY)]
        q = q.set_index("forecast").reindex([f for f in KEY if f in set(q.forecast)])
        q.index = [label(f) for f in q.index]
        print(NETWORKS[net].label, "-", metric, f"{lm} min", region, "(difference to the radar extrapolation)")
        print(q[["estimate", "low", "high"]].round(3), "\n")

Gothenburg (OpenMRG + OpenMRG2), JJA 2015 - CSI_1 60 min domain (difference to the radar extrapolation)
                                                    estimate    low   high
R: persistence                                        -0.140 -0.177 -0.110
R: extrapolation (own motion)                          0.346  0.260  0.418
R: S-PROG (own motion)                                 0.007 -0.002  0.016
R: STEPS mean                                          0.014 -0.004  0.035
C: extrapolation (own motion)                         -0.134 -0.200 -0.082
C: extrapolation (radar motion)                       -0.087 -0.155 -0.026
P: extrapolation (radar motion)                       -0.139 -0.191 -0.079
CP: extrapolation (radar motion)                      -0.086 -0.153 -0.026
CP: STEPS mean                                        -0.089 -0.170 -0.025
RC: extrapolation (own motion)                        -0.109 -0.171 -0.056
RP: extrapolation (own motion)                        -0.116 -0.158 -0.

## Figures

Written by `run.py report` (shown from the files, not re-drawn here):

![CSI by lead time](../results/figures/csi_by_lead.png)

![Next-hour RMSE at the gauges against the radar extrapolation](../results/figures/gauge_rmse_vs_extrapolation.png)